In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
df=pd.read_csv("TP.csv")

In [18]:
df.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10
0,0,1.0,147.0,2.0,50.08315,2.84104,1.46320,4.3591,2.0,-1.0,-1.0,13.5
1,1,1.0,148.0,2.0,50.38612,3.02972,1.88680,4.3591,2.0,-1.0,-1.0,13.6
2,2,1.0,149.0,2.0,50.69421,3.08086,0.51134,4.3591,2.0,-1.0,-1.0,13.7
3,3,1.0,150.0,2.0,51.00658,3.12377,0.42912,4.3591,2.0,-1.0,-1.0,13.8
4,4,1.0,151.0,2.0,51.32203,3.15445,0.30681,4.3591,2.0,-1.0,-1.0,13.9


In [19]:
df.columns

Index(['Unnamed: 0', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10'], dtype='object')

In [20]:
df.drop(columns="Unnamed: 0",inplace=True)

In [21]:
df.drop(columns="10",inplace=True)

In [22]:
df=df.rename(columns={"0":"Vehicle ID","1":"Frame Id","2":"Lane Id","3":"LocalY","4":"Mean Speed","5":"Mean Acceleration","6":"Vehicle Length","7":"Vehicle Class Id","8":"Follower Id","9":"Leader Id"})

In [23]:
df.columns

Index(['Vehicle ID', 'Frame Id', 'Lane Id', 'LocalY', 'Mean Speed',
       'Mean Acceleration', 'Vehicle Length', 'Vehicle Class Id',
       'Follower Id', 'Leader Id'],
      dtype='object')

In [ ]:
import numpy as np
import pandas as pd
from math import isfinite

G = 9.81
CS_THRESH = 0.05 * G  # 0.4905 m/s^2

def linear_fit_slope_intercept(t, v):
    t = np.asarray(t, dtype=float); v = np.asarray(v, dtype=float)
    if len(t) < 2 or np.all(t == t[0]):
        return 0.0, float(v.mean()) if len(v) else 0.0, 0.0
    A = np.vstack([t, np.ones_like(t)]).T
    # least squares slope/intercept
    m, b = np.linalg.lstsq(A, v, rcond=None)[0]
    # SSE (optional, for merge cost)
    sse = np.sum((v - (m*t + b))**2)
    return m, b, sse

def bottom_up_segment(times, speeds, avg_seg_len_s=0.5, fps=10):
    """Extended Bottom-Up segmentation with same-slope merging (paper §3.1)."""
    t = np.asarray(times, dtype=float)
    v = np.asarray(speeds, dtype=float)
    n = len(v)
    if n < 2:
        return [(0, n-1, 0.0)]  # start, end, slope

    total_T = (t[-1] - t[0])
    target_segments = max(1, int(np.ceil(total_T / avg_seg_len_s))) if total_T > 0 else 1

    # starting with primitive segments between consecutive points
    segs = [(i, i+1) for i in range(n-1)]
    def merge_cost(i):
        s = segs[i]; s_next = segs[i+1]
        a, b = s[0], s_next[1]
        _, _, sse = linear_fit_slope_intercept(t[a:b+1], v[a:b+1])
        return sse

    costs = [merge_cost(i) for i in range(len(segs)-1)]

    # merging until target number reached
    while len(segs) > target_segments and len(segs) > 1:
        k = int(np.argmin(costs))
        # merging segs[k] and segs[k+1]
        new_seg = (segs[k][0], segs[k+1][1])
        segs[k:k+2] = [new_seg]
        # updatating costs around k
        costs = []
        for i in range(len(segs)-1):
            costs.append(merge_cost(i))

    # computing slopes for segments
    out = []
    for a, b in segs:
        m, _, _ = linear_fit_slope_intercept(t[a:b+1], v[a:b+1])
        out.append((a, b, m))

    # extended step: merging consecutive segments with same regime sign (paper)
    merged = []
    for seg in out:
        if not merged:
            merged.append(seg)
            continue
        a0, b0, m0 = merged[-1]
        a1, b1, m1 = seg
        def regime_sign(m):
            if abs(m) <= CS_THRESH: return 0
            return 1 if m > 0 else -1
        if regime_sign(m0) == regime_sign(m1):
            # merge
            a = a0; b = b1
            m_new, _, _ = linear_fit_slope_intercept(t[a:b+1], v[a:b+1])
            merged[-1] = (a, b, m_new)
        else:
            merged.append(seg)
    return merged

def dtw_timegap(t_f, v_f, t_l, v_l):
    """DTW alignment path and per-follower-point time gap tau (seconds)."""
    vf = np.asarray(v_f, dtype=float)
    vl = np.asarray(v_l, dtype=float)
    nf, nl = len(vf), len(vl)
    # cost matrix formation
    C = np.abs(vf[:, None] - vl[None, :])
    D = np.full((nf+1, nl+1), np.inf)
    D[0, 0] = 0.0
    # DP
    for i in range(1, nf+1):
        for j in range(1, nl+1):
            D[i, j] = C[i-1, j-1] + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    # backtracking
    i, j = nf, nl
    path = []
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        step = np.argmin([D[i-1, j], D[i, j-1], D[i-1, j-1]])
        if step == 0:
            i -= 1
        elif step == 1:
            j -= 1
        else:
            i -= 1; j -= 1
    path.reverse()
    # mapping follower index to (possibly multiple) leader indices on path; average j if multiple
    from collections import defaultdict
    map_j = defaultdict(list)
    for i_idx, j_idx in path:
        map_j[i_idx].append(j_idx)
    tau = np.full(nf, np.nan)
    for i_idx, js in map_j.items():
        j_mean = int(np.round(np.mean(js)))
        tau[i_idx] = t_f[i_idx] - t_l[j_mean]
    # dropping nans (unmatched ends)
    mask = np.isfinite(tau)
    return tau[mask], mask

def classify_segments(times, speeds, cf_mask, segs):
    """Apply method of slopes with ±0.05g and standstill==exactly zero speed."""
    t = np.asarray(times, dtype=float)
    v = np.asarray(speeds, dtype=float)
    labels = np.array([""]*len(v), dtype=object)
    for a, b, m in segs:
        # segment stats
        v_seg = v[a:b+1]
        mean_v = float(np.mean(v_seg))
        is_const = (abs(m) <= CS_THRESH)
        # per paper: standstill only if zero speed exactly
        seg_cf = np.round(cf_mask[a:b+1].mean()) >= 0.5  # majority voting for this segment
        if seg_cf:  # CF section: A, F, D, S
            if is_const and mean_v == 0.0:
                lab = "S"
            elif is_const:
                lab = "F"
            elif m > CS_THRESH:
                lab = "A"
            else:  # m < -CS_THRESH
                lab = "D"
        else:     # FF section: Fa, C
            if is_const:
                lab = "C"
            elif m > CS_THRESH:
                lab = "Fa"
            else:
                # FF should not have decel; fallback to C as FF decel is excluded in paper
                lab = "C"
        labels[a:b+1] = lab
    return labels

def pravt_label_one_pair(df_pair, fps=10, avg_seg_len_s=0.5, mu_lim=5.0, sigma_lim=1.5):
    """df_pair: rows for one follower contiguous span with a constant Leader Id."""
    dfp = df_pair.sort_values("Frame Id").copy()
    # time in seconds from frame index
    t = (dfp["Frame Id"] - dfp["Frame Id"].min()) / float(fps)
    v_f = dfp["Mean Speed"].to_numpy()
    # leader series aligned by frame
    v_l = dfp["Mean Speed_leader"].to_numpy()
    t_l = t.to_numpy()  # same frame grid assumption
    # Stage I: segmentation of follower
    segs = bottom_up_segment(t.to_numpy(), v_f, avg_seg_len_s=avg_seg_len_s, fps=fps)
    # Stage II: DTW -> tau -> threshold T
    tau, keep = dtw_timegap(t.to_numpy(), v_f, t_l, v_l)
    if len(tau) == 0:
        dfp["Driving Regime"] = "Unknown"
        return dfp
    mu = float(np.mean(tau))
    sigma = float(np.std(tau, ddof=0))
    T = mu if (mu > mu_lim or sigma > sigma_lim) else (mu + 2.0*sigma)
    cf_mask = np.zeros(len(v_f), dtype=bool)
    cf_mask[np.where(keep)[0]] = (tau <= T)
    # Stage III: method of slopes
    labels = classify_segments(t.to_numpy(), v_f, cf_mask, segs)
    dfp["Driving Regime"] = labels
    return dfp

def pravt_label_all(df, fps=10, avg_seg_len_s=0.5, mu_lim=5.0, sigma_lim=1.5):
    """Assumes you already self-merged to attach leader columns (as in your snippet)."""
    out = []
    for vid, g in df.groupby("Vehicle ID"):
        # split on runs where Leader Id is constant (leader can change over time)
        leader_series = g["Leader Id"].astype("Int64").to_numpy()
        idx = g.index.to_numpy()
        # find change points
        splits = [0]
        for k in range(1, len(leader_series)):
            if leader_series[k] != leader_series[k-1]:
                splits.append(k)
        splits.append(len(g))
        # per constant-leader span
        g_sorted = g.sort_values("Frame Id")
        arr = g_sorted.to_numpy()
        for s, e in zip(splits[:-1], splits[1:]):
            part = g_sorted.iloc[s:e]
            if part.empty: 
                continue
            out.append(pravt_label_one_pair(part, fps=fps, avg_seg_len_s=avg_seg_len_s,
                                            mu_lim=mu_lim, sigma_lim=sigma_lim))
    if not out:
        return df.assign(Driving_Regime="Unknown")
    return pd.concat(out, ignore_index=True)


In [25]:
import pandas as pd

# Assume your raw dataframe is df
# Columns: 'Vehicle ID', 'Frame Id', 'Lane Id', 'LocalY',
#          'Mean Speed', 'Mean Acceleration', 'Vehicle Length',
#          'Vehicle Class Id', 'Follower Id', 'Leader Id'

# Self-merge: attach leader's LocalY, Mean Speed, Vehicle Length
df_merged = df.merge(
    df[["Vehicle ID", "Frame Id", "LocalY", "Mean Speed", "Vehicle Length"]],
    left_on=["Leader Id", "Frame Id"],
    right_on=["Vehicle ID", "Frame Id"],
    how="left",
    suffixes=("", "_leader")
)

# Drop the duplicate "Vehicle ID_leader" column if you don’t need it
df_merged.drop(columns=["Vehicle ID_leader"], inplace=True)


In [27]:
df_labeled = pravt_label_all(df_merged, fps=10, avg_seg_len_s=0.5, mu_lim=5.0, sigma_lim=1.5)

In [28]:
df_labeled.head()

,Vehicle ID,Frame Id,Lane Id,LocalY,Mean Speed,Mean Acceleration,Vehicle Length,Vehicle Class Id,Follower Id,Leader Id,LocalY_leader,Mean Speed_leader,Vehicle Length_leader,Driving Regime
0,1.0,147.0,2.0,50.08315,2.84104,1.46320,4.3591,2.0,-1.0,-1.0,NaN,NaN,NaN,A
1,1.0,148.0,2.0,50.38612,3.02972,1.88680,4.3591,2.0,-1.0,-1.0,NaN,NaN,NaN,F
2,1.0,149.0,2.0,50.69421,3.08086,0.51134,4.3591,2.0,-1.0,-1.0,NaN,NaN,NaN,F
3,1.0,150.0,2.0,51.00658,3.12377,0.42912,4.3591,2.0,-1.0,-1.0,NaN,NaN,NaN,F
4,1.0,151.0,2.0,51.32203,3.15445,0.30681,4.3591,2.0,-1.0,-1.0,NaN,NaN,NaN,F


In [32]:
df_labeled["Driving Regime"].value_counts()

Driving Regime
F     375548
C     279269
D     157441
A     153904
Fa     80141
S       9498
Name: count, dtype: int64

In [31]:
for i in range(1,8):
    df_i=df_labeled[df_labeled["Lane Id"]==i]
    df_i.to_csv(f"Lane_{i}.csv")